In [1]:
import math
from typing import Optional, Tuple

import torch
import torch.utils.checkpoint
from torch import nn

from transformers.activations import ACT2FN
from transformers.utils import logging
from transformers import LlamaConfig, LlamaForCausalLM, LlamaTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = LlamaTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

#input_text = "Hello, how are you today?"
#inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
#outputs = model.generate(**inputs, max_length=50)
#print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [2]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): 

In [3]:
original_layers = []
original_layers.append(model.model.layers[0].self_attn.q_proj)
original_layers.append(model.model.layers[0].self_attn.k_proj)
original_layers.append(model.model.layers[0].self_attn.v_proj)
original_layers.append(model.model.layers[0].self_attn.o_proj)
original_layers.append(model.model.layers[0].mlp.gate_proj)
original_layers.append(model.model.layers[0].mlp.up_proj)
original_layers.append(model.model.layers[0].mlp.down_proj)

compress_layers = []
compress_layers.append(model.model.layers[0].self_attn.q_proj)
compress_layers.append(model.model.layers[0].self_attn.k_proj)
compress_layers.append(model.model.layers[0].self_attn.v_proj)
compress_layers.append(model.model.layers[0].self_attn.o_proj)
compress_layers.append(model.model.layers[0].mlp.gate_proj)
compress_layers.append(model.model.layers[0].mlp.up_proj)
compress_layers.append(model.model.layers[0].mlp.down_proj)

In [5]:
"""
activation_data_per_layer = [None] * len(original_layers)

def get_activation_hook(index):
    def hook(module, input, output):
        # Save input tensor of the layer (usually a tuple, take first)
        activation_data_per_layer[index] = input[0].detach().cpu()
        print(f"Activation captured for layer {index}, shape: {activation_data_per_layer[index].shape}")
    return hook

hooks = []
for idx, layer in enumerate(original_layers):
    hooks.append(layer.register_forward_hook(get_activation_hook(idx)))

input_text = "The cat sat on the mat."
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

# Remove hooks after collection
for h in hooks:
    h.remove()

for i, act in enumerate(activation_data_per_layer):
    if act is None:
        print(f"No activations recorded for layer {i}")
    else:
        print(f"Layer {i} activation shape: {act.shape}")
"""

'\nactivation_data_per_layer = [None] * len(original_layers)\n\ndef get_activation_hook(index):\n    def hook(module, input, output):\n        # Save input tensor of the layer (usually a tuple, take first)\n        activation_data_per_layer[index] = input[0].detach().cpu()\n        print(f"Activation captured for layer {index}, shape: {activation_data_per_layer[index].shape}")\n    return hook\n\nhooks = []\nfor idx, layer in enumerate(original_layers):\n    hooks.append(layer.register_forward_hook(get_activation_hook(idx)))\n\ninput_text = "The cat sat on the mat."\ninputs = tokenizer(input_text, return_tensors="pt").to(model.device)\n\nwith torch.no_grad():\n    outputs = model(**inputs)\n\n# Remove hooks after collection\nfor h in hooks:\n    h.remove()\n\nfor i, act in enumerate(activation_data_per_layer):\n    if act is None:\n        print(f"No activations recorded for layer {i}")\n    else:\n        print(f"Layer {i} activation shape: {act.shape}")\n'

In [4]:
# Step 1: Prepare empty lists to collect activations
activation_data_per_layer = [[] for _ in original_layers]

# Step 2: Define hook that appends inputs (first 10 tokens only)
def extract_first_tensor(obj):
    if isinstance(obj, torch.Tensor):
        print("Found hook")
        return obj
    elif isinstance(obj, (tuple, list)):
        print("Recursively finding hook")
        for item in obj:
            t = extract_first_tensor(item)
            if t is not None:
                return t
    return None

def get_activation_hook(index):
    def hook(module, input, output):
        print(f"Hook triggered for layer {index}")
        tensor = extract_first_tensor(input)
        print(f"Extracted tensor type: {type(tensor)}")
        if tensor is None:
            print(f"Layer {index}: No tensor found in output, skipping.")
            return
        # Defensive check
        if not isinstance(tensor, torch.Tensor):
            print(f"Layer {index}: Extracted object is not a tensor, skipping.")
            return
        activation_data_per_layer[index].append(tensor.detach().cpu())
        print(f"Layer {index}: captured activation shape {tensor.shape}")
    return hook


#for i, layer in enumerate(model.model.layers):
#    layer.register_forward_hook(get_activation_hook(i))

# Step 3: Register hooks
hooks = []
for idx, layer in enumerate(original_layers):
    hooks.append(layer.register_forward_hook(get_activation_hook(idx)))

# Step 4: Feed text samples into the model
input_texts = [
    "The cat sat on the mat.",
    "A quick brown fox jumps over the lazy dog.",
    "Hello world!"
]

for text in input_texts:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        _ = model(**inputs)

# Step 5: Remove hooks
for h in hooks:
    h.remove()
for i, layer in enumerate(original_layers):
    print(f"Hooking layer {i}: {layer} (type: {type(layer)})")
    
for i, layer_data in enumerate(activation_data_per_layer):
    print(f"Layer {i} collected batches: {[t.shape for t in layer_data]}")

# Step 6: Flatten activations to shape [tokens, dim] per layer
activation_data_per_layer = [
    torch.cat(layer_data, dim=1).reshape(-1, layer_data[0].shape[-1])
    for layer_data in activation_data_per_layer
]

Hook triggered for layer 0
Recursively finding hook
Found hook
Extracted tensor type: <class 'torch.Tensor'>
Layer 0: captured activation shape torch.Size([1, 8, 2048])
Hook triggered for layer 1
Recursively finding hook
Found hook
Extracted tensor type: <class 'torch.Tensor'>
Layer 1: captured activation shape torch.Size([1, 8, 2048])
Hook triggered for layer 2
Recursively finding hook
Found hook
Extracted tensor type: <class 'torch.Tensor'>
Layer 2: captured activation shape torch.Size([1, 8, 2048])
Hook triggered for layer 3
Recursively finding hook
Found hook
Extracted tensor type: <class 'torch.Tensor'>
Layer 3: captured activation shape torch.Size([1, 8, 2048])
Hook triggered for layer 4
Recursively finding hook
Found hook
Extracted tensor type: <class 'torch.Tensor'>
Layer 4: captured activation shape torch.Size([1, 8, 2048])
Hook triggered for layer 5
Recursively finding hook
Found hook
Extracted tensor type: <class 'torch.Tensor'>
Layer 5: captured activation shape torch.Size(

In [5]:
def test_svd_compression_old(original_layer: nn.Linear, svd_layer_1: nn.Linear, svd_layer_2: nn.Linear, batch_size=1):
    # Generate random input
    input_dim = original_layer.in_features
    x = torch.randn(batch_size, input_dim)
    x = x.to(dtype=torch.bfloat16, device=svd_layer_1.weight.device)

    with torch.no_grad():
        original_output = original_layer(x)
        # Pass input through the two compressed layers sequentially
        compressed_output = svd_layer_2(svd_layer_1(x))

    diff = torch.norm(original_output - compressed_output)
    rel_error = diff / torch.norm(original_output)

    print(f"Difference norm: {diff.item():.6f}")
    print(f"Relative error: {rel_error.item():.6f}")

    return rel_error.item()

# Example usage:
# Suppose original_linear is your original nn.Linear layer,
# svd_linear_1 and svd_linear_2 are your compressed Linear layers.
# test_svd_compression(original_q_proj, new_1, new_2, batch_size=10)

In [6]:
def truncation_aware_SVD(weight: torch.Tensor, rank: int):
    # Compute full SVD
    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)
    rank = get_energy_rank(S, threshold=0.95)

    U_k = U[:, :rank]
    S_k = S[:rank]
    V_k = Vh[:rank, :] 

    U_perp = U[:, rank:]
    S_perp = S[rank:]
    V_perp = Vh[rank:, :]

    # Residual matrix from truncated part
    R = U_perp @ torch.diag(S_perp) @ V_perp

    # Correction terms (simplified)
    # Invert diagonal matrix of singular values
    S_k_inv = torch.diag(1.0 / (S_k + 1e-8))

    delta_U = R @ V_k.T @ S_k_inv
    delta_V = R.T @ U_k @ S_k_inv

    # Apply corrections
    U_hat = U_k + delta_U
    V_hat = V_k + delta_V.T

    # Optional: Re-orthogonalize corrected U and V
    #U_hat, _ = torch.linalg.qr(U_hat)
    #V_hat, _ = torch.linalg.qr(V_hat.T)
    #V_hat = V_hat.T

    return U_hat, S_k, V_hat

def get_energy_rank(S: torch.Tensor, threshold: float = 0.95):
    energy = torch.cumsum(S**2, dim=0)
    total_energy = energy[-1]
    k = torch.searchsorted(energy, total_energy * threshold).item() + 1
    return k

In [7]:
def compute_whitening(X: torch.Tensor):
    # Center the activations
    mean = X.mean(dim=0, keepdim=True)
    X_centered = X - mean

    # Compute covariance matrix
    cov = X_centered.T @ X_centered / (X_centered.shape[0] - 1)

    # Eigen-decomposition / SVD for whitening matrix
    U, S, _ = torch.linalg.svd(cov)
    # Add epsilon for stability
    eps = 1e-5
    D_inv_sqrt = torch.diag(1.0 / torch.sqrt(S + eps))

    whitening_matrix = U @ D_inv_sqrt @ U.T
    return mean, whitening_matrix

def whiten(X: torch.Tensor, mean: torch.Tensor, whitening_matrix: torch.Tensor):
    X_centered = X - mean
    X_white = X_centered @ whitening_matrix
    return X_white


In [8]:
def layer_SVD_old(layer: nn.Linear, old_layer: nn.Linear, activation_data: torch.Tensor, ratio=0.5):
    #weight = layer.weight.data.float()
    #activation = activation_data.float()
    device = layer.weight.device
    dtype = layer.weight.dtype

    weight = layer.weight.data.to(dtype).to(device)
    activation = activation_data.to(dtype).to(device)
    if activation.dim() > 2:
        # Flatten batch and seq dims into one
        X = activation.reshape(-1, activation.shape[-1])
    else:
        X = activation
    X = X - X.mean(dim=0, keepdim=True)

    print(f"activation shape: {activation.shape}")
    print(f"X shape: {X.shape}")

    #cov = X.T @ X / X.shape[0]
    cov = (X.T @ X / X.shape[0]).float()
    eigvals, eigvecs = torch.linalg.eigh(cov)

    # Avoid numerical instability by adding epsilon
    print(f"eigvals min: {eigvals.min()}, max: {eigvals.max()}")

    eps = 1e-3
    safe_eigvals = torch.clamp(eigvals, min=eps)
    S_inv_sqrt = torch.diag(1.0 / torch.sqrt(safe_eigvals))
    whitening_matrix = eigvecs @ S_inv_sqrt @ eigvecs.T
    whitening_matrix = whitening_matrix.to(dtype).to(device)

    W_whitened = weight @ whitening_matrix

    # Cast W_whitened to float32 before SVD
    W_whitened_f32 = W_whitened.float()  # <-- cast to float32 here
    print(f"W_whitened_f32 stats: min={W_whitened_f32.min()}, max={W_whitened_f32.max()}, mean={W_whitened_f32.mean()}")
    print(f"Any NaNs? {torch.isnan(W_whitened_f32).any()}")
    print(f"Any Infs? {torch.isinf(W_whitened_f32).any()}")

    U, S, Vh = torch.linalg.svd(W_whitened_f32, full_matrices=False)
    # Cast results back to original dtype
    U = U.to(dtype)
    S = S.to(dtype)
    Vh = Vh.to(dtype)
    #U, S, Vh = torch.linalg.svd(W_whitened, full_matrices=False)
    total_energy = (S**2).sum()
    energy_cutoff = ratio * total_energy

    running_energy = 0.0
    rank = 0
    for s in S:
        running_energy += s**2
        rank += 1
        if running_energy >= energy_cutoff:
            break

    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]

    input_dim = weight.shape[1]
    output_dim = weight.shape[0]

    """
    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)
    k = int(len(S) * ratio)
    U_reduced = U[:, :k]
    S_reduced = S[:k]
    Vh_reduced = Vh[:k, :]
    weight_approx = U_reduced @ torch.diag(S_reduced) @ Vh_reduced

    # Shapes:
    input_dim = weight.shape[1]
    output_dim = weight.shape[0]

    mean, whitening_matrix = compute_whitening(activation_data)
    W_tilde = weight @ whitening_matrix
    
    #rank = int(len(weight) * ratio)
    rank = get_energy_rank(S, threshold=0.95)

    U_hat, S_k, Vh_hat = truncation_aware_SVD(W_tilde, rank)
    """

    new1_weight = (torch.diag(S_k) @ Vh_k).to(layer.weight.dtype)
    new2_weight = U_k.to(layer.weight.dtype)

    new_1 = nn.Linear(input_dim, rank, bias=False).to(device=device, dtype=dtype)
    #new_1.weight.data = (Vh_k @ whitening_matrix).to(dtype=torch.bfloat16, device=new_1.weight.device)
    #new_1.weight.data = Vh_hat.to(dtype=torch.bfloat16, device=new_1.weight.device)

    new_2 = nn.Linear(rank, output_dim, bias=False).to(device=device, dtype=dtype)
    #new_2.weight.data = (U_k @ torch.diag(S_k)).to(dtype=torch.bfloat16, device=new_2.weight.device)
    #new_2.weight.data = (U_hat @ torch.diag(S_k)).to(dtype=torch.bfloat16, device=new_2.weight.device)

    new_1.weight.data.copy_(new1_weight)
    new_2.weight.data.copy_(new2_weight)

    #layer = compressed_layer
    test_svd_compression_old(old_layer, new_1, new_2, batch_size=10)
    return nn.Sequential(new_1, new_2)

    

In [9]:
def test_svd_compression(original_layer, compressed_model, batch_size=10):
    # Create random input batch (adjust input size to layer)
    input_dim = original_layer.in_features
    x = torch.randn(batch_size, input_dim)

    # Ensure x dtype matches compressed_model's first parameter dtype (weights)
    target_dtype = next(compressed_model.parameters()).dtype
    x = x.to(target_dtype)

    with torch.no_grad():
        y_orig = original_layer(x)
        y_compressed = compressed_model(x)

    diff = (y_orig - y_compressed).float()  # for stable error computation in float32
    rel_error = diff.norm() / y_orig.float().norm()

    print(f"Relative error after compression: {rel_error.item():.6f}")


In [10]:
import torch
import torch.nn as nn

def layer_SVD(layer, activation_data, ratio=0.5, prev_compressed_layer=None):
    """
    Compress a linear layer using SVD with given activations.
    Automatically applies prev_compressed_layer to activation_data if provided.
    """

    W = layer.weight.data  # (out_dim, in_dim)
    out_dim, in_dim = W.shape

    # Flatten activation data to 2D: (N, in_dim)
    input_tensor = activation_data.view(-1, activation_data.shape[-1])

    # Apply previous compressed layers to activation data if available
    if prev_compressed_layer is not None:
        # Cast input tensor to match dtype of prev_compressed_layer weights (usually bfloat16 or float32)
        target_dtype = next(prev_compressed_layer.parameters()).dtype
        input_tensor = input_tensor.to(target_dtype)

        # Pass through previous compressed layers
        input_tensor = prev_compressed_layer(input_tensor)

    # Center activations (mean subtraction)
    X_prime = input_tensor - input_tensor.mean(dim=0, keepdim=True)  # shape: (N, feature_dim)
    
    # Cast weights and activations to float32 for SVD computations for numerical stability
    W_f = W.float()
    X_prime_f = X_prime.float()

    # SVD on weight matrix
    U, S, Vh = torch.linalg.svd(W_f, full_matrices=False)  # U:(out_dim,out_dim), S:(min_dim,), Vh:(min_dim,in_dim)
    rank = max(1, int(min(U.shape[1], Vh.shape[0]) * ratio))

    # Truncate to chosen rank
    U_k = U[:, :rank]            # (out_dim, rank)
    S_k = S[:rank]               # (rank,)
    V_k = Vh[:rank, :]           # (rank, in_dim)

    # Compute Sigma * V_k, Sigma is diagonal matrix from S_k
    Sigma_V = (S_k.unsqueeze(1) * V_k)  # (rank, in_dim)

    # Check shapes before multiplication
    if Sigma_V.shape[1] != X_prime_f.T.shape[0]:
        raise RuntimeError(f"Shape mismatch: Sigma_V {Sigma_V.shape}, X_prime.T {X_prime_f.T.shape}")

    # Compute pseudoinverse for updating U'
    pseudo_inv = torch.linalg.pinv(Sigma_V @ X_prime_f.T)  # shape: (in_dim, rank) pinv of (rank, N)@(N, in_dim)

    # Update U' using activations
    U_prime = W_f @ X_prime_f.T @ pseudo_inv  # (out_dim, in_dim)@(in_dim, N)@(N, rank) => (out_dim, rank)

    # Compose compressed layer as nn.Sequential of two Linear layers
    compressed_layer = nn.Sequential(
        nn.Linear(in_dim, rank, bias=False),
        nn.Linear(rank, out_dim, bias=False)
    )

    # Set weights for compressed layers
    compressed_layer[0].weight.data = Sigma_V  # (rank, in_dim) matches Linear(in_dim, rank).weight shape: (rank, in_dim)
    compressed_layer[1].weight.data = U_prime  # (out_dim, rank) matches Linear(rank, out_dim).weight shape: (out_dim, rank)

    return compressed_layer

In [11]:
def layer_SVD_baseline(layer: nn.Linear, old_layer: nn.Linear, ratio=0.9):
    device = layer.weight.device
    dtype = layer.weight.dtype

    weight = layer.weight.data.float().to(device)  # Cast to float32 for SVD stability

    # Compute full SVD of weight matrix
    U, S, Vh = torch.linalg.svd(weight, full_matrices=False)

    # Determine rank to keep based on energy ratio
    total_energy = (S**2).sum()
    energy_cutoff = ratio * total_energy

    running_energy = 0.0
    rank = 0
    for s in S:
        running_energy += s**2
        rank += 1
        if running_energy >= energy_cutoff:
            break

    U_k = U[:, :rank]
    S_k = S[:rank]
    Vh_k = Vh[:rank, :]

    input_dim = weight.shape[1]
    output_dim = weight.shape[0]

    # Compose new weights for compressed layers
    new1_weight = torch.diag(S_k) @ Vh_k
    new2_weight = U_k

    # Create two linear layers to replace the original
    new_1 = nn.Linear(input_dim, rank, bias=False).to(device=device, dtype=dtype)
    new_2 = nn.Linear(rank, output_dim, bias=False).to(device=device, dtype=dtype)

    # Copy weights back, cast to original dtype
    new_1.weight.data.copy_(new1_weight.to(dtype))
    new_2.weight.data.copy_(new2_weight.to(dtype))

    test_svd_compression_old(old_layer, new_1, new_2, batch_size=10)
    return nn.Sequential(new_1, new_2)


In [12]:
layer_ranks = [0.5, 0.5, 0.75, 0.5, 0.5, 0.5, 0.2]
for i in range(len(compress_layers)):
    #compressed = layer_SVD(compress_layers[i], activation_data_per_layer[i], ratio=0.5)
    #test_svd_compression(original_layers[i], compressed, batch_size=10)
    compressed = layer_SVD_baseline(compress_layers[i], original_layers[i])
    # Replace the original layer
    if i == 0:
        model.model.layers[0].self_attn.q_proj = compressed
    elif i == 1:
        model.model.layers[0].self_attn.k_proj = compressed
    elif i == 2:
        model.model.layers[0].self_attn.v_proj = compressed
    elif i == 3:
        model.model.layers[0].self_attn.o_proj = compressed
    elif i == 4:
        model.model.layers[0].mlp.gate_proj = compressed
    elif i == 5:
        model.model.layers[0].mlp.up_proj = compressed
    elif i == 6:
        model.model.layers[0].mlp.down_proj = compressed

Difference norm: 33.500000
Relative error: 0.337891
Difference norm: 21.875000
Relative error: 0.291016
Difference norm: 7.562500
Relative error: 0.300781
Difference norm: 17.125000
Relative error: 0.314453
Difference norm: 56.250000
Relative error: 0.318359
Difference norm: 56.750000
Relative error: 0.314453
Difference norm: 57.250000
Relative error: 0.318359


In [13]:
output_compressed = model.generate(**inputs, max_length=50)
print("Compressed output:", tokenizer.decode(output_compressed[0], skip_special_tokens=True))

Compressed output: Hello world!

```

In this example, we're using the `echo` command to print the string "Hello world!" to the console. The `printf` function is used to format the string with the `%s`


In [ ]:
def compute_whitening_matrix(activation_data, eps=1e-5, truncation_ratio=0.9):
    if activation_data.dim() > 2:
        batch_size, seq_len, feat_dim = activation_data.shape
        activation_data = activation_data.reshape(batch_size * seq_len, feat_dim)
        
    X = activation_data.float()
    X_centered = X - X.mean(dim=0, keepdim=True)
    cov = (X_centered.T @ X_centered) / (X_centered.shape[0] - 1)  # [in_dim, in_dim]

    U, S, _ = torch.linalg.svd(cov)  # Full SVD of covariance
    k = max(1, int(truncation_ratio * U.shape[1]))
    U_k = U[:, :k]                   # [in_dim, k]
    S_k = S[:k]                      # [k]

    whitening_mat = U_k @ torch.diag(1.0 / torch.sqrt(S_k + eps))  # [in_dim, k]

    X_whitened = X_centered @ whitening_mat  # [N, k]
    cov_whitened = (X_whitened.T @ X_whitened) / (X_whitened.shape[0] - 1)
    #print(f"cov_whitened (should be ~I):\n{cov_whitened}")

    return whitening_mat, k, U_k, S_k

In [77]:
def compress_layer_with_whitening(layer: nn.Linear, activation_data: torch.Tensor, ratio=0.5):
    if activation_data.dim() > 2:
        batch_size, seq_len, feat_dim = activation_data.shape
        activation_data = activation_data.reshape(batch_size * seq_len, feat_dim)
    
    target_dtype = layer.weight.dtype
    W = layer.weight.data.float()  # [out_dim, in_dim]
    out_dim, in_dim = W.shape

    # Step 1: Compute whitening matrix from activations
    whitening_mat, k, U_k, S_k = compute_whitening_matrix(activation_data, truncation_ratio=ratio)
    # whitening_mat shape: [in_dim, k]
    #print("whitening_mat.shape:", whitening_mat.shape)
    #print("activation_data.shape before whitening:", activation_data.shape)

    # Step 2: Project weights into whitened space: W_whitened = W @ U_k  [out_dim, k]
    #W_whitened = W @ U_k  # [out_dim, k]
    #print("W.shape:", W.shape)
    #print("whitening_mat.shape:", whitening_mat.shape)
    W_whitened = W @ whitening_mat

    whitened = activation_data @ whitening_mat.to(target_dtype)  # [N, k]
    cov = (whitened.T @ whitened) / (whitened.shape[0] - 1)
    print(cov)  # Should be close to identity

    # Step 3: Truncated SVD on W_whitened (shape: out_dim x k)
    U_w, S_w, Vh_w = torch.linalg.svd(W_whitened, full_matrices=False)
    #rank = max(1, int(min(U_w.shape[1], Vh_w.shape[0]) * ratio))
    rank = U_w.shape[1]

    U_k_svd = U_w[:, :rank]          # [out_dim, rank]
    S_k_svd = S_w[:rank]             # [rank]
    V_k_svd = Vh_w[:rank, :]         # [rank, k]

    U_k_svd = U_k @ torch.diag(S_k)
    # Compose Sigma * V_k
    Sigma_V = torch.diag(S_k_svd) @ V_k_svd   # [rank, k]

    # Step 4: Build compressed layers

    # Whitening layer: projects from in_dim -> k (fixed, no bias)
    whitening_layer = nn.Linear(in_dim, k, bias=False)
    whitening_layer.weight.data = whitening_mat.T.to(target_dtype)  # Convert to original dtype
    #whitening_layer.weight.data = whitening_mat.T  # PyTorch Linear weight shape [out_features, in_features]
    #print(f"W.shape = {W.shape}")           # (out_dim, in_dim)
    #print(f"activations.shape = {activation_data.shape}") # (N, in_dim)
    #print(f"whitening_mat.shape = {whitening_mat.shape}")


    # Low-rank factorization layers:
    # First layer: k -> rank
    low_rank_1 = nn.Linear(k, rank, bias=False)
    low_rank_1.weight.data = Sigma_V.to(target_dtype)  # [rank, k]

    # Second layer: rank -> out_dim
    low_rank_2 = nn.Linear(rank, out_dim, bias=False)
    low_rank_2.weight.data = U_k_svd.to(target_dtype)  # [out_dim, rank]

    # Compose sequential layer
    compressed_layer = nn.Sequential(
        whitening_layer,
        low_rank_1,
        low_rank_2
    )

    return compressed_layer

In [78]:
def test_svd_compression(original_layer, compressed_layer, batch_size=10):
    input_dim = original_layer.in_features
    dtype = next(original_layer.parameters()).dtype  # use original dtype
    print(f"Testing compression for layer with input dim {input_dim}")

    x = torch.randn(batch_size, input_dim, dtype=dtype)
    print(f"Input x shape: {x.shape}")

    with torch.no_grad():
        y_orig = original_layer(x)
        y_compressed = compressed_layer(x)

    rel_error = (y_orig - y_compressed).float().norm() / y_orig.float().norm()
    print(f"Relative error after compression: {rel_error.item():.6f}")


In [79]:
def verify_activation_shape(layer: nn.Linear, activation_data: torch.Tensor, layer_idx=None):
    # Flatten activation data if needed
    if activation_data.dim() > 2:
        batch_size, seq_len, feat_dim = activation_data.shape
        activation_data = activation_data.reshape(batch_size * seq_len, feat_dim)

    out_dim, in_dim = layer.weight.shape
    act_feat_dim = activation_data.shape[-1]

    layer_name = f"Layer {layer_idx}" if layer_idx is not None else "Layer"

    print(f"{layer_name} weight shape: [out_dim={out_dim}, in_dim={in_dim}]")
    print(f"{layer_name} activation shape: {activation_data.shape}")

    if act_feat_dim != in_dim:
        raise ValueError(f"{layer_name}: Activation feature dimension ({act_feat_dim}) does NOT match "
                         f"layer input dimension ({in_dim}). Please check your activation data collection.")
    else:
        print(f"{layer_name}: Activation data shape matches layer input dimension ✅")


In [80]:
compressed_layers = []
for i, layer in enumerate(original_layers):
    print(f"\nCompressing original layer {i}...")
    
    activations = activation_data_per_layer[i]  # raw activations only (do not transform)
    verify_activation_shape(layer, activations, layer_idx=i)

    # Compress the original layer using whitening + SVD
    compressed = compress_layer_with_whitening(layer, activations, ratio=0.5)

    # Save for later (if needed)
    compressed_layers.append(compressed)

    # (Optional) test it on some example input
    test_svd_compression(layer, compressed, batch_size=10)
    if i == 0:
        model.model.layers[0].self_attn.q_proj = compressed
    elif i == 1:
        model.model.layers[0].self_attn.k_proj = compressed
    elif i == 2:
        model.model.layers[0].self_attn.v_proj = compressed
    elif i == 3:
        model.model.layers[0].self_attn.o_proj = compressed
    elif i == 4:
        model.model.layers[0].mlp.gate_proj = compressed
    elif i == 5:
        model.model.layers[0].mlp.up_proj = compressed
    elif i == 6:
        model.model.layers[0].mlp.down_proj = compressed



Compressing original layer 0...
Layer 0 weight shape: [out_dim=2048, in_dim=2048]
Layer 0 activation shape: torch.Size([25, 2048])
Layer 0: Activation data shape matches layer input dimension ✅
cov_whitened (should be ~I):
tensor([[ 1.0000e+00,  9.3555e-07,  9.2294e-08,  ..., -2.4411e-06,
         -3.3792e-06, -2.6525e-06],
        [ 9.3555e-07,  9.9999e-01,  2.9502e-07,  ...,  9.5981e-07,
         -2.0058e-06, -1.3086e-06],
        [ 9.2294e-08,  2.9502e-07,  9.9999e-01,  ...,  5.2078e-07,
          1.3166e-06,  9.6242e-07],
        ...,
        [-2.4411e-06,  9.5981e-07,  5.2078e-07,  ...,  1.0713e-11,
          5.7726e-12,  6.4532e-12],
        [-3.3792e-06, -2.0058e-06,  1.3166e-06,  ...,  5.7726e-12,
          2.1931e-11,  1.1274e-11],
        [-2.6525e-06, -1.3086e-06,  9.6242e-07,  ...,  6.4532e-12,
          1.1274e-11,  1.7655e-11]])
tensor([[ 1.4453,  0.4043, -0.1060,  ...,  0.1147,  0.3281, -0.0598],
        [ 0.4043,  1.3672, -0.0957,  ...,  0.1035,  0.2969, -0.0540],
    

RuntimeError: mat1 and mat2 shapes cannot be multiplied (10x256 and 1024x2048)

In [ ]:
"""prev_compressed_layers = []
prev_compressed_model = None

for i, (layer, activations) in enumerate(zip(original_layers, activation_data_per_layer)):
    print(f"\nCompressing layer {i}...")

    # If previous compressed layers exist, transform activations through them first
    if prev_compressed_model is not None:
        # Convert activations to the right dtype for previous compressed model
        activations = activations.to(next(prev_compressed_model.parameters()).dtype)
        with torch.no_grad():
            activations = prev_compressed_model(activations)

    compressed = compress_layer_with_whitening(layer, activations, ratio=0.9)

    test_svd_compression(layer, compressed, batch_size=10)

    prev_compressed_layers.append(compressed)
    prev_compressed_model = nn.Sequential(*prev_compressed_layers)

    # Replace original model weights with compressed layer for inference if needed
    # (Example for layer 0 self-attn q_proj)
    if i == 0:
        model.model.layers[0].self_attn.q_proj = compressed
    elif i == 1:
        model.model.layers[0].self_attn.k_proj = compressed
    elif i == 2:
        model.model.layers[0].self_attn.v_proj = compressed
    elif i == 3:
        model.model.layers[0].self_attn.o_proj = compressed
    elif i == 4:
        model.model.layers[0].mlp.gate_proj = compressed
    elif i == 5:
        model.model.layers[0].mlp.up_proj = compressed
    elif i == 6:
        model.model.layers[0].mlp.down_proj = compressed"""

'prev_compressed_layers = []\nprev_compressed_model = None\n\nfor i, (layer, activations) in enumerate(zip(original_layers, activation_data_per_layer)):\n    print(f"\nCompressing layer {i}...")\n\n    # If previous compressed layers exist, transform activations through them first\n    if prev_compressed_model is not None:\n        # Convert activations to the right dtype for previous compressed model\n        activations = activations.to(next(prev_compressed_model.parameters()).dtype)\n        with torch.no_grad():\n            activations = prev_compressed_model(activations)\n\n    compressed = compress_layer_with_whitening(layer, activations, ratio=0.9)\n\n    test_svd_compression(layer, compressed, batch_size=10)\n\n    prev_compressed_layers.append(compressed)\n    prev_compressed_model = nn.Sequential(*prev_compressed_layers)\n\n    # Replace original model weights with compressed layer for inference if needed\n    # (Example for layer 0 self-attn q_proj)\n    if i == 0:\n       